In [1]:
import pickle
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

In [2]:
# Train models for ALL formats in one run.
# Requires that data-extraction.ipynb has already produced the *_features.pkl files.

import json
from pathlib import Path

FORMAT_CONFIGS = {
    "IPL": {"features_in": "ipl_features.pkl", "model_out": "pipe_ipl.pkl"},
    "ODI": {"features_in": "odi_features.pkl", "model_out": "pipe_odi.pkl"},
    "T20": {"features_in": "t20_features.pkl", "model_out": "pipe_t20.pkl"},
    "Test": {"features_in": "test_features.pkl", "model_out": "pipe_test.pkl"},
}

meta_path = Path("metadata.json")
meta = {}
if meta_path.exists():
    try:
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
    except Exception:
        meta = {}

# Migrate legacy flat metadata into per-format structure under IPL
if "teams" in meta and "IPL" not in meta:
    meta = {"IPL": {"teams": meta.get("teams", []), "cities": meta.get("cities", [])}}

REQUIRED_COLUMNS = {
    "batting_team",
    "bowling_team",
    "city",
    "current_score",
    "balls_left",
    "wickets_left",
    "crr",
    "last_five",
    "innings_total",
}

def make_one_hot_encoder():
    try:
        return OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore")
    except TypeError:
        return OneHotEncoder(sparse=False, drop="first", handle_unknown="ignore")

results = {}

for fmt, cfg in FORMAT_CONFIGS.items():
    features_path = Path(cfg["features_in"])
    if not features_path.exists():
        print(f"[{fmt}] Skipping: missing {features_path}")
        continue

    with open(features_path, "rb") as f:
        df = pickle.load(f)

    if not isinstance(df, pd.DataFrame) or df.empty:
        print(f"[{fmt}] Skipping: {features_path} did not contain a non-empty DataFrame")
        continue

    missing_cols = sorted(REQUIRED_COLUMNS - set(df.columns))
    if missing_cols:
        print(f"[{fmt}] Skipping: missing required columns {missing_cols}")
        continue

    print(f"[{fmt}] Loaded features: {df.shape}")

    # Update metadata for this format
    meta[fmt] = {
        "teams": sorted(df["batting_team"].unique().tolist()),
        "cities": sorted(df["city"].unique().tolist()),
    }

    X = df[[
        "batting_team",
        "bowling_team",
        "city",
        "current_score",
        "balls_left",
        "wickets_left",
        "crr",
        "last_five",
    ]]
    y = df["innings_total"]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

    categorical_cols = ["batting_team", "bowling_team", "city"]
    preprocess = ColumnTransformer(
        transformers=[(
            "cat",
            make_one_hot_encoder(),
            categorical_cols,
        )],
        remainder="passthrough",
    )

    model = RandomForestRegressor(n_estimators=300, random_state=1, n_jobs=-1)
    pipe = Pipeline(steps=[("preprocess", preprocess), ("scale", StandardScaler()), ("model", model)])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    print(f"[{fmt}] R2: {r2:.4f} | MAE: {mae:.2f}")

    out_model = cfg["model_out"]
    with open(out_model, "wb") as f:
        pickle.dump(pipe, f)
    print(f"[{fmt}] Saved model: {out_model}")

    results[fmt] = {"model_out": out_model, "r2": float(r2), "mae": float(mae)}

meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
print(f"Updated {meta_path}")

results

[IPL] Loaded features: (144131, 9)
[IPL] R2: 0.9098 | MAE: 4.77
[IPL] Saved model: pipe_ipl.pkl
[ODI] Loaded features: (721704, 9)
[ODI] R2: 0.9667 | MAE: 4.03
[ODI] Saved model: pipe_odi.pkl
[T20] Loaded features: (367252, 9)
[T20] R2: 0.9389 | MAE: 5.05
[T20] Saved model: pipe_t20.pkl
[Test] Loaded features: (536913, 9)
[Test] R2: 0.9895 | MAE: 2.96
[Test] Saved model: pipe_test.pkl
Updated metadata.json


{'IPL': {'model_out': 'pipe_ipl.pkl',
  'r2': 0.9097916119684353,
  'mae': 4.774861511859812},
 'ODI': {'model_out': 'pipe_odi.pkl',
  'r2': 0.9667431714885439,
  'mae': 4.028722646666731},
 'T20': {'model_out': 'pipe_t20.pkl',
  'r2': 0.9389115562270136,
  'mae': 5.04910291383341},
 'Test': {'model_out': 'pipe_test.pkl',
  'r2': 0.9894981821634508,
  'mae': 2.9599330737994}}

In [3]:
# Legacy cell (disabled). All formats are trained above.
pass

In [4]:
# Legacy cell (disabled). All formats are trained above.
pass

In [5]:
# Legacy cell (disabled).
pass